# Claude vs BYO: Merkle break points between two builds (replayable)

Two builds of the same graph are compared top-down: equal roots stop at once; equal module roots are skipped; unequal
modules are break points, and only inside them do we look at leaves (MATCH / DIFFERS / ONLY_claude / ONLY_byo).
Bytes are the identity for code, prints and labware. **Swap data:** set `A` and `B` to two packs (`python -m baebugs.mvp`
on each tree) or FCOs. With `B = None` the notebook makes the demo BYO build in memory: one part re-scaled and one tray
of their own.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
import copy, hashlib
import pandas as pd
from baebugs import aok, mvp, compare
from baebugs.explore import merkle_root

A = 'site/mvp/mvp.json'   # swap: Claude's (or any) pack
B = None                  # swap: the BYO pack path; None -> demo BYO built below
a = mvp.load(A)

## The demo BYO build (only used when `B is None`)

In [2]:
def byo_demo(pack):
    b = copy.deepcopy(pack)
    part = next(x for x in b['leaves'] if x['module'] == 'lab-parts' and x['path'].endswith('.stl'))
    part['sha256'] = hashlib.sha256(b'their STL, scaled 105%').hexdigest()       # same path, different bytes
    b['leaves'].append({'path': 'print/byo/their_tray.stl', 'module': 'lab-parts', 'kind': 'print', 'claim': 'OWN',
                        'licence': 'their licence', 'status': 'built', 'sha256': hashlib.sha256(b'their tray').hexdigest()})
    for x in b['leaves']:
        if x['module'] == 'lab-parts': x['leaf'] = aok.leaf_hash(x)
    lp = [x['leaf'] for x in b['leaves'] if x['module'] == 'lab-parts']
    b['modules']['lab-parts'] = {**b['modules']['lab-parts'], 'root': merkle_root(sorted(lp))}
    b['root'] = merkle_root(sorted(m['root'] for m in b['modules'].values()))
    return b, part['path']

b, changed = (json.loads(Path(B).read_text()), None) if B else byo_demo(a)
print('A', a['root'][:16], '| B', b['root'][:16], '| changed part:', changed)

A 548b4d8fee9f1d1b | B 0270efeaad927808 | changed part: print/filters/stl/CANVAS-10.stl


## Compare

In [3]:
r = compare.compare(a, b, 'claude', 'byo')
print('identical:', r['identical'], '| break points:', r['break_points'])
pd.DataFrame([{'module': m, 'verdict': d['verdict'], 'match': d.get('MATCH', '-'), 'differs': len(d.get('DIFFERS', [])),
               'only_claude': len(d.get('ONLY_claude', [])), 'only_byo': len(d.get('ONLY_byo', []))} for m, d in r['modules'].items()])

identical: False | break points: ['lab-parts']


,module,verdict,match,differs,only_claude,only_byo
0,custody-ledgers,SAME,-,0,0,0
1,daisy-protocols,SAME,-,0,0,0
2,daisy-results,SAME,-,0,0,0
3,dashboards,SAME,-,0,0,0
4,docs,SAME,-,0,0,0
5,kaggle-kernels,SAME,-,0,0,0
6,knowledge-graph,SAME,-,0,0,0
7,lab-parts,BREAK,105,1,0,1
8,mvp-site,SAME,-,0,0,0
9,notebooks,SAME,-,0,0,0


In [4]:
for m in r['break_points']:
    d = r['modules'][m]
    print(m, '\n  DIFFERS', d['DIFFERS'], '\n  ONLY_byo', d['ONLY_byo'], '\n  ONLY_claude', d['ONLY_claude'])

lab-parts 
  DIFFERS ['print/filters/stl/CANVAS-10.stl'] 
  ONLY_byo ['print/byo/their_tray.stl'] 
  ONLY_claude []


## Same build on both sides stops at the root

In [5]:
assert compare.compare(a, copy.deepcopy(a))['identical']

## The committed example (`docs/byo_compare_example.json`)
This is the result Byron's demo shows; re-run `baebugs-graph compare <their pack>` to regenerate it.

In [6]:
ex = json.loads(Path('docs/byo_compare_example.json').read_text())
print('break points', ex['break_points'], '| lab-parts match', ex['modules']['lab-parts'].get('MATCH'),
      '| differs', ex['modules']['lab-parts'].get('DIFFERS'), '| only byo', ex['modules']['lab-parts'].get('ONLY_byo'))

break points ['lab-parts'] | lab-parts match 105 | differs ['print/parts/stl/arm_grip_handle.stl'] | only byo ['print/parts/stl/byo_flat_tray.stl']


## BYO with only some files: an FCO is enough to prove what matches

In [7]:
fco = mvp.fco_of(['LICENSE', 'baebugs/compare.py'])
[(x['path'], x['verdict']) for x in mvp.prove(a, fco)['rows']]

[('LICENSE', 'MATCH'), ('baebugs/compare.py', 'MATCH')]